<a href="https://colab.research.google.com/github/shivainlabs/Introduction-to-Deep-Learning-and-GenAI/blob/main/Week%2006/GAN_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms # used for image preprocessing
import torchvision
import os
import matplotlib.pyplot as plt

In [30]:
sample_dir = 'samples'
if not os.path.exists(sample_dir):
  os.makedirs(sample_dir)

In [31]:
latent_size = 64
hidden_size = 256
image_size = 784
num_epochs = 100

batch_size = 100

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [32]:
device

device(type='cuda')

In [33]:
def load_mnist_data(batch_size):
  transform = transforms.Compose([
      transforms.ToTensor(),
      transforms.Normalize(mean=[0.5],std=[0.5])
  ])

  mnist = torchvision.datasets.FashionMNIST(
      root = "./data/",
      train = True,
      transform = transform,
      download = True
  )

  data_loader = torch.utils.data.DataLoader(
      dataset = mnist,
      batch_size = batch_size,
      shuffle=True
  )

  return data_loader


In [34]:
data_loader = load_mnist_data(batch_size)

In [35]:
class Generator(nn.Module):
  def __init__(self,latent_size,hidden_size,image_size):
    super().__init__()

    self.model = nn.Sequential(
        nn.Linear(latent_size,hidden_size),
        nn.LeakyReLU(0.2),

        nn.Linear(hidden_size,hidden_size),
        nn.LeakyReLU(0.2),

        nn.Linear(hidden_size,image_size),
        nn.Tanh()
    )

  def forward(self,z):
    return self.model(z)

generator = Generator(latent_size,hidden_size,image_size).to(device)
generator


Generator(
  (model): Sequential(
    (0): Linear(in_features=64, out_features=256, bias=True)
    (1): LeakyReLU(negative_slope=0.2)
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): LeakyReLU(negative_slope=0.2)
    (4): Linear(in_features=256, out_features=784, bias=True)
    (5): Tanh()
  )
)

> ```latent_size = dimension of z```





In [36]:
class Discriminator(nn.Module):
  def __init__(self,image_size,hidden_size):
    super().__init__()

    self.model = nn.Sequential(
        nn.Linear(image_size,hidden_size),
        nn.LeakyReLU(0.2),

        nn.Linear(hidden_size,hidden_size),
        nn.LeakyReLU(0.2),

        nn.Linear(hidden_size,1),
        nn.Sigmoid()
    )

  def forward(self,x):
    return self.model(x)

discriminator = Discriminator(image_size,hidden_size).to(device)
discriminator

Discriminator(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=256, bias=True)
    (1): LeakyReLU(negative_slope=0.2)
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): LeakyReLU(negative_slope=0.2)
    (4): Linear(in_features=256, out_features=1, bias=True)
    (5): Sigmoid()
  )
)

In [37]:
criterion = nn.BCELoss()

d_optimizer_fixed = torch.optim.Adam(discriminator.parameters(),lr=0.0002,betas=(0.5,0.999))
g_optimizer_fixed = torch.optim.Adam(generator.parameters(),lr=0.0002,betas=(0.5,0.999))


In [38]:
def denorm(x):
  out = (x+1)/2
  return out.clamp(0,1)

def reset_grad():
  d_optimizer_fixed.zero_grad()
  g_optimizer_fixed.zero_grad()

In [39]:

print(len(data_loader.dataset))# number of images
print(batch_size) # Images per batch


# number of batches
print(len(data_loader))
print(len(data_loader.dataset)/batch_size)

60000
100
600
600.0
